In [3]:
import pandas as pd

print("Reading and cleaning datasets from the current directory...\n")

def process_wb_data(filename, value_name):
    try:
        df = pd.read_csv(filename, skiprows=4)
        if 'Country Code' not in df.columns:
            df = pd.read_csv(filename)
    except:
        df = pd.read_csv(filename)

    # dropping unnecessary columns from World Bank format.
    cols_to_drop = ['Country Name', 'Indicator Name', 'Indicator Code', 'Unnamed: 66', 'Unnamed: 67', 'Unnamed: 68']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # convertint years from columns to rows (Melt to Long Format)
    df_melted = df.melt(id_vars=['Country Code'], var_name='Year', value_name=value_name)

    # converting Year to integer and drop invalid rows
    df_melted['Year'] = pd.to_numeric(df_melted['Year'], errors='coerce')
    df_melted = df_melted.dropna(subset=['Year'])
    df_melted['Year'] = df_melted['Year'].astype(int)

    # standardizing the merge key
    df_melted = df_melted.rename(columns={'Country Code': 'Country_Code'})
    return df_melted

# processing World Bank Datasets.
df_gdp = process_wb_data('gdp_per_capita.csv', 'GDP_Per_Capita')
df_pop = process_wb_data('population.csv', 'Population')
df_alc = process_wb_data('alcohol_consumption.csv', 'Alcohol_Per_Capita')

# processing Traffic Deaths Dataset.
df_traffic = pd.read_csv('traffic_deaths.csv')

# find the exact column name for deaths and standardize it.
traffic_col_name = [col for col in df_traffic.columns if 'Death' in col or 'injuries' in col][0]
df_traffic = df_traffic[['Code', 'Year', traffic_col_name]]
df_traffic.columns = ['Country_Code', 'Year', 'Traffic_Deaths']

# merging all datasets.
print("Merging all datasets on 'Country_Code' and 'Year'...")
df_merged = df_traffic.merge(df_pop, on=['Country_Code', 'Year'], how='inner')
df_merged = df_merged.merge(df_gdp, on=['Country_Code', 'Year'], how='inner')
df_merged = df_merged.merge(df_alc, on=['Country_Code', 'Year'], how='inner')

# Changing traffic deaths into traffic deaths per 100,000 people.
df_merged['Death_Rate_Per_100k'] = (df_merged['Traffic_Deaths'] / df_merged['Population']) * 100000

# track rows before dropna
initial_len = len(df_merged)

# drop out the problematic values.
df_final = df_merged.dropna()
df_final.to_csv('processed_dataset.csv', index=False)


# --- Summary Panel ---

print("\n" + "="*50)
print("             DATA COLLECTION SUMMARY")
print("="*50)

num_countries = df_final['Country_Code'].nunique()
num_years = df_final['Year'].nunique()
expected_rows = num_countries * num_years
actual_rows = len(df_final)

print(f"Total Unique Countries  : {num_countries}")
print(f"Total Years Covered     : {num_years} ({df_final['Year'].min()} - {df_final['Year'].max()})")
print(f"Expected Data Points    : {num_countries} countries * {num_years} years = {expected_rows} rows")
print(f"Actual Clean Rows       : {actual_rows}")
print(f"Data Gaps (Missing)     : {expected_rows - actual_rows} rows")

# rows dropped exactly at the dropna step
rows_lost_to_na = initial_len - actual_rows
print(f"\n[-] Rows dropped due to missing (NaN) values: {rows_lost_to_na}")

# finding Mismatched/Lost Countries due to different data sources.
# (Countries that had Traffic Death data but were lost during Inner Merge or NaN drop)
core_countries = set(df_traffic['Country_Code'].dropna().unique())
final_countries = set(df_final['Country_Code'].unique())
lost_countries = core_countries - final_countries

print(f"[-] Total countries completely lost in the process: {len(lost_countries)}")
if len(lost_countries) > 0:
    sample_lost = list(lost_countries)[:20] # There should be 19 dropped codes, just to make sure it displays maximum of 20 examples
    print(f"    Examples of dropped country codes: {sample_lost}")

print("="*50)
print(f"Success! Cleaned and merged dataset saved as 'processed_dataset.csv'.\n")

Reading and cleaning datasets from the current directory...

Merging all datasets on 'Country_Code' and 'Year'...

             DATA COLLECTION SUMMARY
Total Unique Countries  : 186
Total Years Covered     : 20 (2000 - 2019)
Expected Data Points    : 186 countries * 20 years = 3720 rows
Actual Clean Rows       : 3677
Data Gaps (Missing)     : 43 rows

[-] Rows dropped due to missing (NaN) values: 2323
[-] Total countries completely lost in the process: 19
    Examples of dropped country codes: ['ASM', 'PSE', 'BMU', 'TKL', 'MCO', 'MNP', 'NIU', 'VIR', 'PRI', 'COK', 'GRL', 'GUM', 'SSD', 'SMR', 'OWID_WRL', 'TWN', 'PRK', 'MHL', 'PLW']
Success! Cleaned and merged dataset saved as 'processed_dataset.csv'.

